# Análise do LegibiliMeter × Avaliação da turma

Este notebook lê os CSVs gerados pela ferramenta e **fundamenta a validação** do
LegibiliMeter, comparando a nota automática [0–10] com o julgamento humano de
legibilidade da turma.

**Fontes de dados** (pasta `eval/`):

| Arquivo | Conteúdo |
| :--- | :--- |
| `final-results.csv` | nota da ferramenta × média da turma, por snippet (com resíduo) |
| `class-scores.csv` | média/mediana da turma por snippet (36 participantes, escala 1–5) |
| `control-results.csv` | smoke test nos exemplos sintéticos de controle (ideal/problemático) |

A verdade humana vem da inspeção de código da disciplina (`inspecao-codigo-s2`):
40 snippets avaliados por 36 alunos. **S24 foi excluído** por ser Java não
parseável → análise sobre **39 snippets**.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# Funciona rodando da raiz do projeto ou da pasta analysis/
EVAL = Path('eval') if Path('eval').exists() else Path('..') / 'eval'

results = pd.read_csv(EVAL / 'final-results.csv')
classes = pd.read_csv(EVAL / 'class-scores.csv')
control = pd.read_csv(EVAL / 'control-results.csv')
print('Snippets avaliados:', len(results))
results.head()

## 1. Correlação com a avaliação humana

Critério da proposta (Seção 3.2): a ferramenta é válida se **Spearman ρ > 0,6**
frente ao ranking humano. Pearson mede concordância linear; Spearman, concordância
de ordenação (mais robusto a diferenças de escala 0–10 vs. 1–5).

In [ ]:
tool = results['toolScore']
human = results['classMean']

r, _ = pearsonr(tool, human)
rho, _ = spearmanr(tool, human)

print(f'N        = {len(results)} snippets')
print(f'Pearson  r   = {r:.3f}')
print(f'Spearman rho = {rho:.3f}')
print(f'Criterio rho > 0.6 -> {"ATENDIDO" if rho > 0.6 else "NAO atendido"}')

In [ ]:
x = results['classMeanScaled0_10']
y = results['toolScore']

plt.figure(figsize=(7, 7))
plt.scatter(x, y, zorder=3)
for _, row in results.iterrows():
    plt.annotate(row['snippet'], (row['classMeanScaled0_10'], row['toolScore']),
                 fontsize=7, alpha=0.6)

lims = [0, 10]
plt.plot(lims, lims, 'k--', alpha=0.5, label='concordancia perfeita')
m, b = np.polyfit(x, y, 1)
xs = np.array(lims)
plt.plot(xs, m * xs + b, 'r-', alpha=0.7, label=f'regressao (y={m:.2f}x+{b:.2f})')

plt.xlabel('Nota da turma (escala 0-10)')
plt.ylabel('Nota da ferramenta (0-10)')
plt.title(f'LegibiliMeter vs. turma  (rho={rho:.2f}, r={r:.2f})')
plt.xlim(0, 10); plt.ylim(0, 10)
plt.legend(); plt.grid(alpha=0.3)
plt.show()

**Leitura.** A nuvem sobe da esquerda para a direita: snippets que a turma achou
mais legíveis tendem a receber nota mais alta da ferramenta. Com ρ ≈ 0,64 a
ferramenta **atende ao critério da proposta** — captura a ordenação humana de
legibilidade, embora a correlação seja moderada (não perfeita). A reta de
regressão fica acima da diagonal na ponta baixa: a ferramenta tende a ser
**mais generosa** que a turma com os piores casos (ver resíduos abaixo).

## 2. Análise de resíduos — onde a ferramenta diverge

Resíduo = nota da ferramenta − nota da turma (ambas 0–10). Positivo = ferramenta
generosa demais; negativo = severa demais.

In [ ]:
res = results.sort_values('residual')
colors = ['#d62728' if v > 0 else '#1f77b4' for v in res['residual']]

plt.figure(figsize=(12, 5))
plt.bar(res['snippet'], res['residual'], color=colors)
plt.axhline(0, color='k', lw=0.8)
plt.ylabel('Residuo (ferramenta - turma)')
plt.title('Residuos por snippet  (vermelho=generosa, azul=severa)')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

cols = ['snippet', 'toolScore', 'classMeanScaled0_10', 'residual']
print('== Ferramenta mais GENEROSA (resíduo +) ==')
print(res[cols].tail(5).to_string(index=False))
print('\n== Ferramenta mais SEVERA (resíduo -) ==')
print(res[cols].head(5).to_string(index=False))

**Dois vieses sistemáticos** explicam a maior parte do erro:

1. **Generosa com código críptico porém "plano"** (resíduo +): `S12`, `S28`,
   `S22` — expressões bit-a-bit/aritméticas densas em uma linha e nomes
   abreviados. A Complexidade Cognitiva atual só conta fluxo de controle
   (`if/for/while`), então não enxerga complexidade de expressão nem nomes ruins.

2. **Severa com aninhamento** (resíduo −): `S05` (aninhado, mas de lógica clara)
   e `S06` (versão com guard clauses). Mesmo após a calibração Nível A (limiar de
   aninhamento 5→6, peso 25%→20%), a ferramenta ainda pune mais que humanos.

O item 1 é o erro dominante e motiva o **Nível B** da calibração (densidade de
operadores + fração de nomes curtos), ainda não implementado.

## 3. Smoke test — exemplos sintéticos de controle

Trechos escritos pelo grupo (sem código de repositório externo) representando os
extremos. Critério: **ideal > 7,0** e **problemático < 4,0**.

In [ ]:
print(control.to_string(index=False))

passed = (control['pass'].astype(str).str.lower() == 'true').sum()
colors = ['#2ca02c' if c == 'ideal' else '#d62728' for c in control['category']]

plt.figure(figsize=(8, 4))
plt.bar(control['sample'], control['toolScore'], color=colors)
plt.axhline(7, color='#2ca02c', ls='--', label='ideal > 7.0')
plt.axhline(4, color='#d62728', ls='--', label='problematico < 4.0')
plt.ylabel('Nota da ferramenta')
plt.title(f'Smoke test: {passed}/{len(control)} na faixa esperada')
plt.ylim(0, 10); plt.legend(); plt.tight_layout()
plt.show()

**Resultado:** 6/6 exemplos na faixa esperada — código sabidamente bom recebe
nota alta e código sabidamente ruim recebe nota baixa. Note que os exemplos
problemáticos têm *todas* as features ruins (aninhamento + parâmetros + nomes +
linhas longas); o caso difícil para a ferramenta continua sendo código com
*apenas algumas* features ruins, como o `S28` da turma (item 2 da §2).

## 4. Efeito da calibração (Nível A)

Comparação dos números antes e depois da recalibração de parâmetros
(ver `eval/DECISOES-CALIBRACAO.md`). O baseline são os valores da proposta
original; o pós-calibração é recomputado das células acima.

In [ ]:
calib = pd.DataFrame({
    'fase':     ['Baseline (proposta)', 'Apos Nivel A'],
    'pearson':  [0.547, round(r, 3)],
    'spearman': [0.532, round(rho, 3)],
})
print(calib.to_string(index=False))

ax = calib.set_index('fase')[['pearson', 'spearman']].plot.bar(figsize=(7, 4))
ax.axhline(0.6, color='k', ls='--', alpha=0.6)
ax.annotate('criterio rho=0.6', (0.5, 0.61), ha='center')
ax.set_ylabel('correlacao'); ax.set_title('Antes x depois da calibracao')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 5. Conclusão

- **Correlação com humanos:** Spearman ρ ≈ 0,64 (> 0,6) e Pearson r ≈ 0,61
  (> 0,5). A ferramenta concorda, de forma moderada e na direção certa, com o
  julgamento de legibilidade da turma.
- **Smoke test:** 6/6 exemplos de controle na faixa esperada.
- **Vieses conhecidos:** generosa com código críptico "plano" (S12/S28/S22);
  ainda algo severa com aninhamento (S05/S06).

**Limitações desta análise:**

- *Tuning in-sample:* calibração e avaliação no mesmo conjunto de 39 snippets →
  possível overfitting; os ajustes foram conservadores.
- *Pendente:* Nível B da calibração (densidade de operadores, nomes curtos) e
  comparação com Complexidade Ciclomática via PMD (critério 3 da Seção 3.2).